# Hazard counter and split by state

1. **Counter:** per AGEB and scenario, how many of 4 hazards (`heat_bio2`, `drought_pet_annual`, `cold_bio6`, `flood_bio13`) have `inciden_median = 1` (≥ median). Maximum = 4. One counter per comparison level:
   `hazard_count_nati`, `hazard_count_state`, `hazard_count_mun`.
2. **Split:** 32 files, one per state (`CVEGEO[:2]`), to share and load into Tableau.
   Exported as CSV (UTF-8, `CVEGEO` as text) and as Parquet.

Input: `climate_ssp_gcm_all_ageb_metrics.parquet` (notebook 02).

In [1]:
from pathlib import Path
import pandas as pd

def find_hazard_dir(start=Path.cwd().resolve()):
    for p in [start, *start.parents]:
        d = p / "data" / "hazard" / "ageb"
        if (d / "climate_ssp_gcm_all_ageb_metrics.parquet").exists():
            return d
    raise FileNotFoundError("Run notebook 02 first")

DIR = find_hazard_dir()
IN_PARQUET = DIR / "climate_ssp_gcm_all_ageb_metrics.parquet"
OUT_FULL = DIR / "climate_ssp_gcm_all_ageb_metrics_count.parquet"
OUT_STATES = DIR / "climate_by_state"

df = pd.read_parquet(IN_PARQUET)
df["CVEGEO"] = df["CVEGEO"].astype(str)
print(df.shape)

(19548240, 74)


## 1. Hazard counter

In [2]:
COUNT_HAZARDS = ["heat_bio2", "drought_pet_annual", "cold_bio6", "flood_bio13"]

for lvl in ["nati", "state", "mun"]:
    cols = [f"{h}_inciden_median_{lvl}" for h in COUNT_HAZARDS]
    df[f"hazard_count_{lvl}"] = df[cols].sum(axis=1).astype("int8")

# Placed next to the identifier columns
ids = ["CVEGEO", "ssp", "gcm", "time_period"]
cnt = [f"hazard_count_{l}" for l in ["nati", "state", "mun"]]
df = df[ids + cnt + [c for c in df.columns if c not in ids + cnt]]

assert df[cnt].min().min() >= 0 and df[cnt].max().max() <= 4
for c in cnt:
    print(c, (df[c].value_counts(normalize=True).sort_index() * 100).round(1).to_dict())

hazard_count_nati {0: 5.6, 1: 27.8, 2: 34.6, 3: 25.1, 4: 7.0}
hazard_count_state {0: 7.2, 1: 23.2, 2: 38.9, 3: 23.4, 4: 7.3}
hazard_count_mun {0: 7.0, 1: 23.9, 2: 32.7, 3: 25.9, 4: 10.4}


## 2. State code and name

In [3]:
STATES = {
 "01":"aguascalientes","02":"baja_california","03":"baja_california_sur","04":"campeche","05":"coahuila",
 "06":"colima","07":"chiapas","08":"chihuahua","09":"ciudad_de_mexico","10":"durango","11":"guanajuato",
 "12":"guerrero","13":"hidalgo","14":"jalisco","15":"mexico","16":"michoacan","17":"morelos","18":"nayarit",
 "19":"nuevo_leon","20":"oaxaca","21":"puebla","22":"queretaro","23":"quintana_roo","24":"san_luis_potosi",
 "25":"sinaloa","26":"sonora","27":"tabasco","28":"tamaulipas","29":"tlaxcala","30":"veracruz","31":"yucatan","32":"zacatecas"}

df.insert(1, "CVE_ENT", df["CVEGEO"].str[:2])
assert set(df.CVE_ENT.unique()) == set(STATES), set(df.CVE_ENT.unique()) ^ set(STATES)
df.groupby("CVE_ENT").CVEGEO.nunique().rename("n_ageb").to_frame().T

CVE_ENT,01,02,03,04,05,06,07,08,09,10,...,23,24,25,26,27,28,29,30,31,32
n_ageb,553,2030,1108,722,2973,601,2693,4745,2453,2583,...,1252,2148,2622,4559,852,3182,765,4295,1879,2914


## 3. Save the full table and split by state

In [4]:
df.to_parquet(OUT_FULL, index=False, compression="zstd")
print(OUT_FULL, f"{OUT_FULL.stat().st_size/1e6:.0f} MB")

(OUT_STATES / "csv").mkdir(parents=True, exist_ok=True)
(OUT_STATES / "parquet").mkdir(parents=True, exist_ok=True)

rows = []
for cve, g in df.groupby("CVE_ENT", observed=True):
    name = f"{cve}_{STATES[cve]}"
    g = g.drop(columns=[]).reset_index(drop=True)
    g.to_csv(OUT_STATES / "csv" / f"{name}.csv", index=False, float_format="%.6g")
    g.to_parquet(OUT_STATES / "parquet" / f"{name}.parquet", index=False, compression="zstd")
    rows.append({"state": name, "rows": len(g), "ageb": g.CVEGEO.nunique(),
                 "csv_MB": round((OUT_STATES / "csv" / f"{name}.csv").stat().st_size / 1e6),
                 "parquet_MB": round((OUT_STATES / "parquet" / f"{name}.parquet").stat().st_size / 1e6)})
summary = pd.DataFrame(rows)
assert summary.rows.sum() == len(df)
summary

/Users/fabianfuentes/git/adatpation_ageb_data/data/hazard/ageb/climate_ssp_gcm_all_ageb_metrics_count.parquet 1343 MB


,state,rows,ageb,csv_MB,parquet_MB
0,01_aguascalientes,132720,553,60,9
1,02_baja_california,487200,2030,223,36
2,03_baja_california_sur,265920,1108,120,18
3,04_campeche,173280,722,78,13
4,05_coahuila,713520,2973,322,52
5,06_colima,144240,601,65,10
6,07_chiapas,646320,2693,287,47
7,08_chihuahua,1138800,4745,515,84
8,09_ciudad_de_mexico,588720,2453,265,38
9,10_durango,619920,2583,279,41


## 4. National CSV

All AGEB in a single CSV (≈19.5 M rows; does not fit in Excel, works in Tableau).

In [5]:
OUT_CSV_NATIONAL = DIR / "climate_ssp_gcm_all_ageb_metrics_count.csv"
df.to_csv(OUT_CSV_NATIONAL, index=False, float_format="%.6g")
print(OUT_CSV_NATIONAL, f"{OUT_CSV_NATIONAL.stat().st_size/1e9:.1f} GB")

/Users/fabianfuentes/git/adatpation_ageb_data/data/hazard/ageb/climate_ssp_gcm_all_ageb_metrics_count.csv 8.8 GB
